Data Loading

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

DATASET_DIR = "/kaggle/input/competitions/icdar-2026-circleid-writer-identification"
PRETRAINED_DIR = "/kaggle/input/datasets/cbdsigmas/tranfer-to-writer-model"

train_1 = pd.read_csv(f"{DATASET_DIR}/train.csv")
additional_path = Path(DATASET_DIR) / "additional_train.csv"
if additional_path.exists():
    train_2 = pd.read_csv(additional_path)
    train_df = pd.concat([train_1, train_2], ignore_index=True)
    del(train_2)
else:
    train_df = train_1.copy()

test_df = pd.read_csv(f"{DATASET_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATASET_DIR}/sample_submission.csv")

del(train_1)
print(f"train_df shape: {train_df.shape}")
print(f"test_df shape: {test_df.shape}")

train_df.sample(min(7, len(train_df)))


train_df shape: (40250, 4)
test_df shape: (5905, 2)


,image_id,image_path,writer_id,pen_id
14939,25173,images/25173.png,W33,5
32316,20744,images/20744.png,-1,1
2332,3890,images/03890.png,W10,6
24432,1447,images/01447.png,W42,3
38850,36811,images/36811.png,-1,8
7482,12646,images/12646.png,W51,4
32657,21594,images/21594.png,-1,6


In [2]:
if "image_path" in train_df.columns:
    train_df["image_path"] = train_df["image_path"].astype(str).map(
        lambda x: x if x.startswith("/") else f"{DATASET_DIR}/{x}"
    )
else:
    train_df["image_path"] = train_df["image_id"].astype(str).map(
        lambda x: f"{DATASET_DIR}/images/{x}.png"
    )

if "image_path" in test_df.columns:
    test_df["image_path"] = test_df["image_path"].astype(str).map(
        lambda x: x if x.startswith("/") else f"{DATASET_DIR}/{x}"
    )
else:
    test_df["image_path"] = test_df["image_id"].astype(str).map(
        lambda x: f"{DATASET_DIR}/images/{x}.png"
    )

train_df["writer_id"] = train_df["writer_id"].astype(str)
unlabeled_mask = train_df["writer_id"].eq("-1")
unlabeled_count = int(unlabeled_mask.sum())

train_df = train_df.loc[~unlabeled_mask].reset_index(drop=True)

writer_ids = sorted(train_df["writer_id"].unique())
writer_to_label = {writer_id: idx for idx, writer_id in enumerate(writer_ids)}
label_to_writer = {idx: writer_id for writer_id, idx in writer_to_label.items()}
NUM_CLASSES = len(writer_ids)

train_df["label"] = train_df["writer_id"].map(writer_to_label)
print("Dropped unlabeled rows:", unlabeled_count)
print("NUM_CLASSES:", NUM_CLASSES)
train_df[["writer_id", "label"]].head()


Dropped unlabeled rows: 5600
NUM_CLASSES: 44


,writer_id,label
0,W27,21
1,W17,15
2,W01,0
3,W17,15
4,W24,19


In [3]:
import os
import torch

XLA_AVAILABLE = False
pl = None
xm = None

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    XLA_AVAILABLE = True
except ImportError:
    pass

USE_TPU = XLA_AVAILABLE and (
    os.environ.get("TPU_NAME") is not None
    or os.environ.get("PJRT_DEVICE") == "TPU"
)

if USE_TPU:
    os.environ.setdefault("PJRT_DEVICE", "TPU")
    device = xm.xla_device()
    print("Using TPU:", device)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", device)
else:
    device = torch.device("cpu")
    print("Using CPU:", device)


Using GPU: cuda


Train/validation split

In [4]:
from sklearn.model_selection import train_test_split

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

print(train_split_df.shape, val_split_df.shape)


(29452, 5) (5198, 5)


Dataset creation

In [5]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class CircleDataset(Dataset):

    def __init__(self, df, img_root, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root = Path(img_root)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row["image_id"]

        label = row["label"]

        return image, label


Image Transformation

In [6]:
from torchvision import transforms

normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

train_tfms = transforms.Compose([

    transforms.RandomResizedCrop(
        224,
        scale=(0.95,1.0),
        ratio=(0.95,1.05)
    ),

    transforms.RandomRotation(12),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03,0.03),
        scale=(0.97,1.03)
    ),

    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08
    ),

    transforms.ToTensor(),

    normalize
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])


Loading Model

In [7]:
import torch.nn as nn
from torchvision.models import resnet18, convnext_tiny

PRETRAINED_PATHS = {
    "resnet": f"{PRETRAINED_DIR}/resnet_best.pth",
    "convnext": f"{PRETRAINED_DIR}/convnext_best.pth",
}


def load_pretrained_backbone(model, model_name):
    ckpt_path = PRETRAINED_PATHS[model_name]

    if not os.path.exists(ckpt_path):
        print(f"Missing pretrained {model_name} checkpoint: {ckpt_path}")
        return model

    checkpoint = torch.load(ckpt_path, map_location="cpu")
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            checkpoint = checkpoint["model_state_dict"]
        elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
            checkpoint = checkpoint["model"]

    checkpoint = {
        key.replace("module.", ""): value
        for key, value in checkpoint.items()
    }

    if model_name == "resnet":
        checkpoint = {
            key: value for key, value in checkpoint.items()
            if not key.startswith("fc.")
        }
    else:
        checkpoint = {
            key: value for key, value in checkpoint.items()
            if not key.startswith("classifier.2")
        }

    missing, unexpected = model.encoder.load_state_dict(checkpoint, strict=False)
    print(f"Loaded pretrained {model_name} from {ckpt_path}")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    return model


class WriterMetricModel(nn.Module):

    def __init__(self, model_name="resnet"):
        super().__init__()

        if model_name == "resnet":
            backbone = resnet18(weights="IMAGENET1K_V1")
            feature_dim = backbone.fc.in_features
            backbone.fc = nn.Identity()
        elif model_name == "convnext":
            backbone = convnext_tiny(weights="IMAGENET1K_V1")
            feature_dim = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        self.encoder = backbone
        self.embedding = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True)
        )
        self.head = nn.Linear(256, NUM_CLASSES)

    def forward(self, x):
        features = self.encoder(x)
        embeddings = self.embedding(features)
        embeddings = nn.functional.normalize(embeddings, dim=1)
        logits = self.head(embeddings)
        return logits, embeddings


def build_model(model_name="resnet"):
    model = WriterMetricModel(model_name)
    model = load_pretrained_backbone(model, model_name)
    return model


Model Training

In [8]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

IS_XLA = str(device).startswith("xla")
TRIPLET_WEIGHT = 0.25
TRIPLET_MARGIN = 0.2


def batch_hard_triplet_loss(embeddings, labels, margin=TRIPLET_MARGIN):
    if embeddings.size(0) < 3:
        return embeddings.new_tensor(0.0)

    triplet_loss = nn.TripletMarginLoss(margin=margin, p=2)
    unique_labels = labels.unique()
    losses = []

    for label in unique_labels:
        pos_idx = torch.where(labels == label)[0]
        neg_idx = torch.where(labels != label)[0]

        if len(pos_idx) < 2 or len(neg_idx) == 0:
            continue

        anchor_embeddings = embeddings[pos_idx]
        positive_distances = torch.cdist(anchor_embeddings, embeddings[pos_idx])
        positive_distances.fill_diagonal_(-1)
        hard_positive = embeddings[pos_idx][positive_distances.argmax(dim=1)]

        negative_distances = torch.cdist(anchor_embeddings, embeddings[neg_idx])
        hard_negative = embeddings[neg_idx][negative_distances.argmin(dim=1)]

        losses.append(triplet_loss(anchor_embeddings, hard_positive, hard_negative))

    if not losses:
        return embeddings.new_tensor(0.0)

    return torch.stack(losses).mean()


def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits, embeddings = model(images)
        ce_loss = F.cross_entropy(logits, labels, label_smoothing=0.05)
        triplet_loss = batch_hard_triplet_loss(embeddings, labels)
        loss = ce_loss + TRIPLET_WEIGHT * triplet_loss

        loss.backward()

        if IS_XLA:
            xm.optimizer_step(optimizer, barrier=True)
            xm.mark_step()
        else:
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [9]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits, _ = model(images)

        preds = logits.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if IS_XLA:
            xm.mark_step()

    return correct / total


In [10]:
from torch.utils.data import DataLoader

BATCH_SIZE = 96
EPOCHS_resnet = 6
EPOCHS_convnext = 4
NUM_WORKERS = 2
PIN_MEMORY = not IS_XLA
LOWER_LR = 1e-4
DIST_THRESHOLD = 0.9

models = []

train_ds = CircleDataset(train_split_df, DATASET_DIR, train_tfms)
val_ds = CircleDataset(val_split_df, DATASET_DIR, val_tfms)
prototype_ds = CircleDataset(train_split_df, DATASET_DIR, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

prototype_loader = DataLoader(
    prototype_ds,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


def build_prototypes(model, loader, device):
    model.eval()
    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            _, embeddings = model(images)
            all_embeddings.append(embeddings.detach().cpu())
            all_labels.append(labels.detach().cpu())

    all_embeddings = torch.cat(all_embeddings, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    prototype_embeddings = []
    prototype_labels = []
    for label in sorted(all_labels.unique().tolist()):
        prototype_embeddings.append(all_embeddings[all_labels == label].mean(dim=0))
        prototype_labels.append(label)

    return {
        "embeddings": torch.stack(prototype_embeddings),
        "labels": np.array(prototype_labels)
    }


for model_name in ["resnet", "convnext"]:

    print(f"Training {model_name}")

    model = build_model(model_name).to(device)
    best_path = os.path.join(SAVE_DIR, f"writer_{model_name}_best.pth")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LOWER_LR,
        weight_decay=1e-4
    )

    if model_name == "resnet":
        EPOCHS = EPOCHS_resnet
    else:
        EPOCHS = EPOCHS_convnext

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_acc = 0.0

    for epoch in range(EPOCHS):

        if IS_XLA:
            epoch_train_loader = pl.MpDeviceLoader(train_loader, device)
            epoch_val_loader = pl.MpDeviceLoader(val_loader, device)
        else:
            epoch_train_loader = train_loader
            epoch_val_loader = val_loader

        train_loss = train_epoch(model, epoch_train_loader, optimizer, device)

        val_acc = validate(model, epoch_val_loader, device)

        print(f"{model_name} Epoch {epoch} loss {train_loss:.4f} val_acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            print(f"Saved best {model_name} to {best_path}")

        scheduler.step()

    print(f"Best {model_name} val_acc: {best_val_acc:.4f}")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    prototypes = build_prototypes(model, prototype_loader, device)
    models.append((model, prototypes))


Training resnet
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 214MB/s]


Loaded pretrained resnet from /kaggle/input/datasets/cbdsigmas/tranfer-to-writer-model/resnet_best.pth
Missing keys: []
Unexpected keys: []


100%|██████████| 306/306 [03:12<00:00,  1.59it/s]


resnet Epoch 0 loss 3.8074 val_acc 0.2168
Saved best resnet to /kaggle/working/writer_resnet_best.pth


...

convnext Epoch 1 loss 3.6458 val_acc 0.3855
Saved best convnext to /kaggle/working/writer_convnext_best.pth


100%|██████████| 306/306 [08:42<00:00,  1.71s/it]


convnext Epoch 2 loss 3.5226 val_acc 0.4733
Saved best convnext to /kaggle/working/writer_convnext_best.pth


100%|██████████| 306/306 [08:41<00:00,  1.71s/it]


convnext Epoch 3 loss 3.4506 val_acc 0.5219
Saved best convnext to /kaggle/working/writer_convnext_best.pth
Best convnext val_acc: 0.5219


Inference

In [11]:
test_ds = CircleDataset(test_df, DATASET_DIR, val_tfms, is_test=True)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

distance_preds = []

for model, prototypes in models:

    model.eval()

    fold_distances = []

    prototype_embeddings = prototypes["embeddings"].to(device)

    with torch.no_grad():

        if IS_XLA:
            inference_loader = pl.MpDeviceLoader(test_loader, device)
        else:
            inference_loader = test_loader

        for images, _ in inference_loader:

            images = images.to(device)
            _, embeddings = model(images)
            distances = torch.cdist(embeddings, prototype_embeddings)
            fold_distances.append(distances.cpu().numpy())

            if IS_XLA:
                xm.mark_step()

    distance_preds.append(np.concatenate(fold_distances))

mean_distances = np.mean(distance_preds, axis=0)
labels = mean_distances.argmin(axis=1)
min_distances = mean_distances.min(axis=1)
prototype_labels = models[0][1]["labels"]

writer_preds = [
    label_to_writer[int(prototype_labels[label])] if distance <= DIST_THRESHOLD else "-1"
    for label, distance in zip(labels, min_distances)
]


Submission

In [12]:
submission = sample_submission.copy()
submission["writer_id"] = writer_preds
submission.to_csv("submission.csv", index=False)

submission.head()


,image_id,writer_id
0,v2_000010c19b698fe39922ef49ff26ce16,W39
1,v2_000f8ac7d2e0ecf1d336c60fe7f34789,W10
2,v2_001a7e63cc91c92d259c2bc43572824f,W42
3,v2_001dd98a324ee7d88952f2e1d0c29674,W48
4,v2_002e3bc1d82ac5121dccf470ceed4422,W13
